# Merchandise Store: source exploration

The stakeholder is a Head of Growth. Before defining conversion, retention or revenue opportunities, this notebook establishes the observable event inventory and checks that the local extraction agrees with BigQuery.

Source: `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`, 1 November 2020 through 31 January 2021. Google's sample is obfuscated and may be internally inconsistent; these results describe the sample rather than current store performance. [Dataset documentation](https://developers.google.com/analytics/bigquery/web-ecommerce-demo-dataset).

Run from the repository root after `python scripts/setup_db.py`. The notebook reads the raw database without changing it. Outputs are produced by execution, not manually entered.

In [1]:
from pathlib import Path
import json
import duckdb
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "ga4.duckdb").exists():
    raise FileNotFoundError("Build ga4.duckdb from the repository root before executing this notebook.")
con = duckdb.connect(str(ROOT / "ga4.duckdb"), read_only=True)
con.execute("SET memory_limit='512MB'")
con.execute("SET threads=1")
con.execute("SET TimeZone='UTC'")

def read_json(relative):
    return json.loads((ROOT / relative).read_text(encoding="utf-8"))

source = read_json("docs/verification/source_coverage.json")[0]
profile = pd.DataFrame(read_json("docs/verification/extraction_source_profile.json")["rows"])

## Coverage

The raw event count establishes extraction completeness. Distinct pseudonymous IDs are not verified people, and an event named `session_start` is not itself a session model. Property dates remain separate from UTC event time.

In [2]:
coverage = con.sql("""
SELECT COUNT(*) AS event_count,
       COUNT(DISTINCT user_pseudo_id) AS distinct_user_ids_raw,
       COUNT(DISTINCT event_date) AS day_count,
       COUNT(DISTINCT event_name) AS event_type_count,
       MIN(event_date) AS first_property_date,
       MAX(event_date) AS last_property_date,
       (SELECT COUNT(*) FROM items_flat) AS item_occurrences
FROM events_flat
""").df()
for field in ["event_count", "distinct_user_ids_raw", "day_count", "event_type_count"]:
    assert int(coverage.iloc[0][field]) == int(source[field]), field
assert int(coverage.iloc[0]["item_occurrences"]) == int(profile["item_rows"].astype(int).sum())
coverage

,event_count,distinct_user_ids_raw,day_count,event_type_count,first_property_date,last_property_date,item_occurrences
0,4295584,270154,92,17,2020-11-01,2021-01-31,3982732


## Event inventory

This inventory determines which funnel steps can be observed. Every event-type count is compared with the independently saved BigQuery inventory. A later funnel will require an explicit unit of analysis, step order and time window.

In [3]:
inventory = con.sql("""
SELECT event_name, COUNT(*) AS event_count
FROM events_flat
GROUP BY event_name
ORDER BY event_count DESC, event_name
""").df()
expected_inventory = {r["event_name"]: int(r["event_count"])
                      for r in read_json("docs/verification/event_inventory.json")}
assert dict(zip(inventory["event_name"], inventory["event_count"])) == expected_inventory
inventory

,event_name,event_count
0,page_view,1350428
1,user_engagement,1058721
2,scroll,493072
3,view_item,386068
4,session_start,354970
5,first_visit,257462
6,view_promotion,190104
7,add_to_cart,58543
8,begin_checkout,38757
9,select_item,31007


## Monthly extraction controls

Item occurrences include every item-bearing event type. They are not units sold. Purchase-event counts and their revenue sum are extraction controls, before validating and deduplicating transaction IDs.

In [4]:
monthly = con.sql("""
WITH events AS (
    SELECT substr(source_table_suffix, 1, 6) AS source_month,
           COUNT(*) AS event_rows,
           COUNT(DISTINCT event_date) AS property_dates,
           COUNT(*) FILTER (WHERE event_name = 'purchase') AS purchase_events,
           SUM(purchase_revenue_usd) FILTER (WHERE event_name = 'purchase') AS raw_purchase_revenue_usd
    FROM events_flat GROUP BY source_month
), items AS (
    SELECT substr(source_table_suffix, 1, 6) AS source_month, COUNT(*) AS item_rows
    FROM items_flat GROUP BY source_month
)
SELECT * FROM events JOIN items USING (source_month) ORDER BY source_month
""").df()
controls = monthly.merge(profile, on="source_month", suffixes=("_local", "_source"), validate="one_to_one")
for field in ["event_rows", "property_dates", "purchase_events", "item_rows"]:
    assert (controls[field + "_local"].astype(int) == controls[field + "_source"].astype(int)).all(), field
assert (abs(controls["raw_purchase_revenue_usd_local"].astype(float)
            - controls["raw_purchase_revenue_usd_source"].astype(float)) < .005).all()
monthly

,source_month,event_rows,property_dates,purchase_events,raw_purchase_revenue_usd,item_rows
0,202011,1472712,30,2054,144260.0,1096733
1,202012,1612725,31,2434,160555.0,1849850
2,202101,1210147,31,1204,57350.0,1036149


## Raw metadata quality

These counts establish decisions for the analytical model. Exact full-record repeats remain separate occurrences. Missing integer session IDs, unexpected string-valued IDs and invalid transaction identifiers must be accounted for rather than silently discarded.

In [5]:
con.sql("""
SELECT COUNT(*) FILTER (WHERE source_duplicate_ordinal > 1) AS exact_source_repeat_occurrences,
       COUNT(*) FILTER (WHERE ga_session_id IS NULL) AS missing_integer_session_id,
       COUNT(*) FILTER (WHERE ga_session_id_text IS NOT NULL) AS string_session_id_present,
       COUNT(*) FILTER (WHERE ga_session_number IS NULL) AS missing_integer_session_number
FROM events_flat
""").df()

,exact_source_repeat_occurrences,missing_integer_session_id,string_session_id_present,missing_integer_session_number
0,0,0,0,0


In [6]:
con.sql("""
SELECT COUNT(*) AS purchase_events,
       COUNT(DISTINCT transaction_id) AS distinct_raw_transaction_ids,
       COUNT(*) FILTER (WHERE transaction_id IS NULL) AS null_transaction_id,
       COUNT(*) FILTER (WHERE transaction_id = '') AS blank_transaction_id,
       COUNT(*) FILTER (WHERE transaction_id IN ('<Other>', '(data deleted)', '(not set)')) AS placeholder_transaction_id,
       SUM(purchase_revenue_usd) AS raw_purchase_event_revenue_usd
FROM events_flat WHERE event_name = 'purchase'
""").df()

,purchase_events,distinct_raw_transaction_ids,null_transaction_id,blank_transaction_id,placeholder_transaction_id,raw_purchase_event_revenue_usd
0,5692,4452,23,0,883,362165.0


## Placeholder profile

The saved build report profiles every text column in both raw tables. The table below displays only fields with at least one missing or placeholder value. NULL, empty strings and obfuscation labels remain separate categories. A missing label does not justify removing its event.

In [7]:
placeholders = pd.DataFrame(read_json("docs/verification/placeholder_profile.json"))
missing_fields = ["null", "blank", "other", "data_deleted", "not_set", "whitespace_only"]
placeholders["missing_or_placeholder"] = placeholders[missing_fields].sum(axis=1)
placeholders.loc[placeholders["missing_or_placeholder"] > 0].sort_values(
    ["table", "missing_or_placeholder", "column"], ascending=[True, False, True]
).reset_index(drop=True)

,table,column,rows,null,blank,other,data_deleted,not_set,whitespace_only,missing_or_placeholder
0,events_flat,ga_session_id_text,4295584,4295584,0,0,0,0,0,4295584
1,events_flat,ga_session_number_text,4295584,4295584,0,0,0,0,0,4295584
2,events_flat,transaction_id,4295584,3549275,0,0,0,741523,0,4290798
3,events_flat,first_source,4295584,0,0,1131013,308695,0,0,1439708
4,events_flat,first_campaign,4295584,0,0,782379,306045,0,0,1088424
5,events_flat,first_medium,4295584,0,0,597482,313917,0,0,911399
6,events_flat,browser,4295584,0,0,114439,0,0,0,114439
7,events_flat,os,4295584,0,0,110323,0,0,0,110323
8,events_flat,country,4295584,0,0,0,0,32208,0,32208
9,events_flat,page_title,4295584,20948,0,0,0,0,0,20948


## Saved build verification

This is the report from the database build, including parent-item reconciliation and timestamp fidelity. It is displayed as saved evidence; the notebook does not claim to rerun every build check. Coverage, all event-type counts and monthly controls above were queried again by this notebook.

In [8]:
verification = read_json("docs/verification/extraction_verification.json")
checks = pd.DataFrame(verification["checks"])
assert checks["passed"].all()
checks

,name,passed,actual,expected
0,event_rows_match_source,True,4295584.0,4295584.0
1,raw_users_match_source,True,270154.0,270154.0
2,property_dates_match_source,True,92.0,92.0
3,event_types_match_source,True,17.0,17.0
4,item_rows_match_source,True,3982732.0,3982732.0
5,event_id_unique,True,0.0,0.0
6,item_occurrence_unique,True,0.0,0.0
7,no_null_event_keys,True,0.0,0.0
8,items_have_parent_events,True,0.0,0.0
9,item_counts_match_each_parent,True,0.0,0.0


## Channels, countries and orders

First-touch source and medium as recorded, the largest countries, and purchase events by transaction-id validity. These counts set the channel grouping, the market split and the order definition in `docs/metric_definitions.md`.

In [9]:
con.sql("""
SELECT first_source, first_medium, COUNT(DISTINCT user_pseudo_id) AS users
FROM events_flat GROUP BY ALL ORDER BY users DESC
""").df()

,first_source,first_medium,users
0,google,organic,103487
1,(direct),(none),75951
2,<Other>,<Other>,51037
3,<Other>,referral,32880
4,shop.googlemerchandisestore.com,referral,26065
5,(data deleted),(data deleted),17948
6,google,cpc,15527
7,<Other>,organic,10095
8,<Other>,(data deleted),393
9,(data deleted),referral,3


In [10]:
con.sql("""
SELECT country, COUNT(DISTINCT user_pseudo_id) AS users, COUNT(*) AS events
FROM events_flat GROUP BY country ORDER BY users DESC LIMIT 12
""").df()

,country,users,events
0,United States,118493,1910483
1,India,25367,391859
2,Canada,20268,324261
3,United Kingdom,8527,133383
4,France,5381,82514
5,Spain,5044,81713
6,Germany,4866,71673
7,China,4620,76159
8,Taiwan,4568,72484
9,Italy,3750,61258


In [11]:
con.sql("""
SELECT CASE WHEN transaction_id IS NULL THEN 'NULL'
            WHEN trim(transaction_id) = '' THEN 'blank'
            WHEN transaction_id IN ('(not set)', '<Other>', '(data deleted)') THEN transaction_id
            ELSE 'valid' END AS transaction_id_status,
       purchase_revenue_usd > 0 AS has_revenue,
       COUNT(*) AS purchase_events, COUNT(DISTINCT transaction_id) AS distinct_ids,
       SUM(purchase_revenue_usd) AS revenue_usd
FROM events_flat WHERE event_name = 'purchase'
GROUP BY ALL ORDER BY ALL
""").df()

,transaction_id_status,has_revenue,purchase_events,distinct_ids,revenue_usd
0,(not set),False,450,1,0.0
1,(not set),True,433,1,30489.0
2,NULL,True,23,0,1328.0
3,valid,True,4786,4451,330348.0


Keying sessions on `ga_session_id` alone merges sessions of different users that started in the same second.

In [12]:
sessions = con.sql("""
SELECT (SELECT COUNT(*) FROM (SELECT DISTINCT user_pseudo_id, ga_session_id FROM events_flat)) AS user_plus_session_id,
       (SELECT COUNT(DISTINCT ga_session_id) FROM events_flat) AS session_id_alone,
       (SELECT COUNT(*) FROM events_flat WHERE event_name = 'session_start') AS session_start_events,
       (SELECT COUNT(*) FROM events_flat WHERE ga_session_id IS NULL) AS events_without_session_id
""").df()
con.close()
sessions

,user_plus_session_id,session_id_alone,session_start_events,events_without_session_id
0,360129,349545,354970,0
